# Train A Policy From Demonstrations
**EXERCISE NOTEBOOK · ArtificialGate · Dr. Barak Or**

**Foundation of Physical AI · Physical AI Essentials, Lesson 3**

**Estimated participant work: 60 minutes.** Complete the marked TODO lines. Run supplied cells unchanged unless a task names a parameter to vary.
Times include reading, coding, reruns, and interpretation; download speed can vary.

**Skills you will practise:** Build masked action chunks; train and execute a policy; diagnose feedback loss and learner-state shift.

**Before you start:** Read the matching book lesson; know NumPy arrays and PyTorch tensors/gradients.

## Your 60-Minute Task Plan
| Task | Estimated time |
|---|---:|
| PAI3-T01 · Set Up The Runtime | 4 min |
| PAI3-T02 · Inspect And Split Episodes | 4 min |
| PAI3-T03 · Build Real Action Chunks | 8 min |
| PAI3-T04 · Normalize Training Data | 6 min |
| PAI3-T05 · Complete Masked Sequence Learning | 7 min |
| PAI3-T06 · Train H=1 And H=4 Policies | 9 min |
| PAI3-T07 · Inspect Action Ambiguity | 4 min |
| PAI3-T08 · Decode And Execute A Chunk | 6 min |
| PAI3-T09 · Evaluate Feedback And State Coverage | 8 min |
| PAI3-T10 · Save And Explain | 4 min |
| **Total** | **60 min** |

### Task PAI3-T01 — Set Up The Runtime · 4 min
**Do:** Run setup with the supplied seed.  
**Produce:** A ready message.  
**Check:** No paid API or GPU is required.

In [ ]:
#@title Setup
import os, sys, subprocess, importlib.util
from importlib.metadata import version, PackageNotFoundError
os.environ.setdefault("SDL_VIDEODRIVER","dummy"); os.environ.setdefault("SDL_AUDIODRIVER","dummy")
os.environ.setdefault("HF_HUB_DISABLE_XET","1")
requirements={"numpy":"numpy>=1.26,<3","pandas":"pandas>=2.2,<3","pyarrow":"pyarrow>=17,<24",
              "matplotlib":"matplotlib>=3.8,<4","torch":"torch>=2.2,<3","huggingface_hub":"huggingface_hub>=0.25,<2",
              "gym_pusht":"gym-pusht==0.1.8","PIL":"Pillow>=10,<13"}
missing=[spec for name,spec in requirements.items() if importlib.util.find_spec(name) is None]
for package,required in {"gym-pusht":"0.1.8","pymunk":"6.11.1"}.items():
    try: installed=version(package)
    except PackageNotFoundError: installed=None
    if installed!=required: missing.append(f"{package}=={required}")
if missing and os.environ.get("AG_LOCAL_TEST")!="1":
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*dict.fromkeys(missing)])
import copy, hashlib, json, math, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from huggingface_hub import hf_hub_download
from IPython.display import display, Image
import gymnasium as gym
import gym_pusht
from PIL import Image as PILImage
assert version("gym-pusht")=="0.1.8" and version("pymunk")=="6.11.1"
torch.set_num_threads(min(4,os.cpu_count() or 1)); DEVICE=torch.device("cpu"); SEED=29
torch.manual_seed(SEED); np.random.seed(SEED)
BLUE,NAVY,CYAN,SILVER="#174BFF","#081638","#17B8D4","#8798B6"
plt.rcParams.update({"axes.spines.top":False,"axes.spines.right":False,"figure.dpi":110,
                     "axes.prop_cycle":plt.cycler(color=[BLUE,CYAN,NAVY,SILVER]),"font.size":10})
OUT=Path("artificialgate_lab3_chunk_outputs"); OUT.mkdir(exist_ok=True)
print("Ready:",torch.__version__,"|",DEVICE)

### Task PAI3-T02 — Inspect And Split Episodes · 4 min
**Do:** Run the loader. Inspect the fixed 100 training episodes and independent validation/test episodes.  
**Produce:** Source hashes and split manifest.  
**Check:** Only two past/current observations enter each policy input.

In [ ]:
#@title Provided code — run this cell
DATASET_ID="lerobot/pusht_keypoints"; REVISION="ace8c161a68bc025c21a5f29f85b86a9a2c5e64b"
FILENAME="data/chunk-000/file-000.parquet"
EXPECTED_SHA256="8b4d978eeeb99254ddead01e79ee70d95a0416e27438ca9438c89a1682598ae0"
path=Path(hf_hub_download(DATASET_ID,FILENAME,repo_type="dataset",revision=REVISION,token=False))
assert hashlib.sha256(path.read_bytes()).hexdigest()==EXPECTED_SHA256,"Dataset integrity check failed."
df=pd.read_parquet(path).sort_values(["episode_index","frame_index"]).reset_index(drop=True)
agent_xy=np.stack(df["observation.state"].to_numpy()).astype("float32")
object_xy=np.stack(df["observation.environment_state"].to_numpy()).astype("float32")
actions=np.stack(df["action"].to_numpy()).astype("float32")
episode_ids=df.episode_index.to_numpy(dtype="int64"); frame_ids=df.frame_index.to_numpy(dtype="int64")
assert agent_xy.shape[1]==2 and object_xy.shape[1]==16 and actions.shape[1]==2
assert np.isfinite(np.concatenate([agent_xy,object_xy,actions],axis=1)).all()
for _,group in df.groupby("episode_index",sort=False):
    assert np.all(np.diff(group.frame_index)>0) and np.all(np.diff(group.timestamp)>0)
observations=np.concatenate([agent_xy,object_xy],axis=1); previous=observations.copy()
same_episode=episode_ids[1:]==episode_ids[:-1]; previous[1:][same_episode]=observations[:-1][same_episode]
features=np.concatenate([previous,observations],axis=1)
shuffled=np.random.default_rng(SEED).permutation(np.unique(episode_ids))
val_ids,test_ids,train_pool=shuffled[:30],shuffled[30:60],shuffled[60:]
TRAIN_EPISODES,HORIZONS=100,[1,4]; LONG_HORIZON=max(HORIZONS)
assert len(HORIZONS)==2 and min(HORIZONS)==1 and LONG_HORIZON>1
train_ids=train_pool[:TRAIN_EPISODES]
train_rows,val_rows,test_rows=[np.isin(episode_ids,ids) for ids in [train_ids,val_ids,test_ids]]
assert len(train_ids)==TRAIN_EPISODES and set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(test_ids) and set(val_ids).isdisjoint(test_ids)
manifest={"dataset":DATASET_ID,"revision":REVISION,"sha256":EXPECTED_SHA256,"file":FILENAME,
          "origin":"simulation demonstrations","license":"Apache-2.0","train":train_ids.tolist(),
          "validation":val_ids.tolist(),"test":test_ids.tolist(),
          "runtime":{p:version(p) for p in ["torch","numpy","pandas","gym-pusht","gymnasium","pymunk"]}}
(OUT/"data_manifest.json").write_text(json.dumps(manifest,indent=2))
print(f"{len(df):,} frames | {len(np.unique(episode_ids))} episodes | input 36 → future 2H actions")

### Task PAI3-T03 — Build Real Action Chunks · 8 min
**Do:** Complete TODO 1: construct future-action labels and within-episode validity masks. Express every chunk element relative to its CURRENT agent anchor.  
**Produce:** H=4 labels and boundary masks.  
**Check:** No label crosses a reset; padded tail elements are masked.

In [ ]:
def make_action_chunks(horizon):
    rows=np.arange(len(df)); labels=np.zeros((len(df),horizon,2),dtype="float32")
    masks=np.zeros((len(df),horizon),dtype="float32")
    for offset in range(horizon):
        future=rows+offset; in_bounds=future<len(df); safe=np.minimum(future,len(df)-1)
        valid=None  # TODO
        assert valid is not None,"TODO 1: bounds, same episode and consecutive frame offset."
        future_displacement=None  # TODO
        assert future_displacement is not None,"TODO 1: future target minus CURRENT anchor."
        labels[valid,offset]=future_displacement; masks[valid,offset]=1
    return labels,masks
chunk_labels,chunk_masks=make_action_chunks(LONG_HORIZON)
episode_ends=np.flatnonzero(np.r_[episode_ids[1:]!=episode_ids[:-1],True])
assert np.all(chunk_masks[:,0]==1) and np.all(chunk_masks[episode_ends,1:]==0)
assert np.allclose(chunk_labels[:,0],actions-agent_xy)
full_row=np.flatnonzero(chunk_masks.sum(1)==LONG_HORIZON)[0]
assert np.allclose(chunk_labels[full_row]+agent_xy[full_row],actions[full_row:full_row+LONG_HORIZON])
display(pd.DataFrame(chunk_masks[episode_ends[:4]],columns=[f"future_{j}" for j in range(LONG_HORIZON)]))
print("PASS: episode-safe future supervision and current-frame anchor.")

### Task PAI3-T04 — Normalize Training Data · 6 min
**Do:** Complete TODO 2 using training rows only.  
**Produce:** Shared normalization for both policies.  
**Check:** Validation/test data do not determine statistics.

In [ ]:
x_mean=None  # TODO
x_std=None  # TODO
assert x_mean is not None and x_std is not None,"TODO 2: use selected training rows only."
assert np.allclose(x_mean,features[train_rows].mean(0))
assert np.allclose(x_std,np.maximum(features[train_rows].std(0),1.0))
valid_labels=chunk_labels[train_rows][chunk_masks[train_rows].astype(bool)]
stats={"x_mean":x_mean,"x_std":x_std,"y_mean":valid_labels.mean(0),"y_std":np.maximum(valid_labels.std(0),1.0)}
x_all=torch.tensor((features-x_mean)/x_std,dtype=torch.float32)
y_all=torch.tensor((chunk_labels-stats["y_mean"])/stats["y_std"],dtype=torch.float32)
m_all=torch.tensor(chunk_masks,dtype=torch.float32)
assert torch.isfinite(x_all).all() and torch.isfinite(y_all).all()
print("PASS: common train-only statistics; padded labels excluded.")

### Task PAI3-T05 — Complete Masked Sequence Learning · 7 min
**Do:** Complete TODO 3: per-step error, masked mean, backpropagation, and update.  
**Produce:** A working sequence objective.  
**Check:** Changing padded targets does not change the loss.

In [ ]:
def masked_mse(prediction,target,valid_mask):
    per_step=None  # TODO
    assert per_step is not None,"TODO 3: squared component error, averaged over x/y."
    loss=None  # TODO
    assert loss is not None,"TODO 3: average over valid future steps only."
    return loss
def learning_step(model,optimizer,xb,yb,mb):
    model.train(); optimizer.zero_grad(set_to_none=True)
    prediction=model(xb); loss=masked_mse(prediction,yb,mb)
    raise NotImplementedError('Complete the TODO learning step')
    raise NotImplementedError('Complete the TODO learning step')
    return float(loss.detach())
boundary=torch.tensor(episode_ends[:4]); targets_at_end=y_all[boundary]; masks_at_end=m_all[boundary]
changed_padding=targets_at_end.clone(); changed_padding[masks_at_end==0]+=10000
zeros=torch.zeros_like(targets_at_end)
assert torch.allclose(masked_mse(zeros,targets_at_end,masks_at_end),masked_mse(zeros,changed_padding,masks_at_end))
print("PASS: padded actions do not alter the sequence loss.")

### Task PAI3-T06 — Train H=1 And H=4 Policies · 9 min
**Do:** Run both models using the same episodes, common initial layers, seed, and optimizer-update budget.  
**Produce:** Learning curves and two actual checkpoints.  
**Check:** Weights change; compare first-action error as the common validation metric.

In [ ]:
#@title Provided code — run this cell
LEARNING_RATE=1e-3; UPDATES=int(os.environ.get("AG_LAB3_UPDATES","2500")); BATCH_SIZE=256
assert UPDATES>0 and LEARNING_RATE>0
class Policy(nn.Module):
    def __init__(self,horizon):
        super().__init__(); self.horizon=horizon
        self.trunk=nn.Sequential(nn.Linear(36,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU())
        self.head=nn.Linear(128,2*horizon)
    def forward(self,x): return self.head(self.trunk(x)).reshape(-1,self.horizon,2)
torch.manual_seed(SEED); initial=Policy(LONG_HORIZON).state_dict()
def initialized_policy(horizon):
    model=Policy(horizon)
    model.trunk.load_state_dict({k.removeprefix("trunk."):v for k,v in initial.items() if k.startswith("trunk.")})
    with torch.no_grad():
        model.head.weight.copy_(initial["head.weight"][:2*horizon]); model.head.bias.copy_(initial["head.bias"][:2*horizon])
    return model.to(DEVICE)
def native_errors(prediction,target,mask):
    delta=(prediction-target)*torch.tensor(stats["y_std"],device=prediction.device)
    first=torch.linalg.vector_norm(delta[:,0],dim=-1).mean().item()
    sequence=((delta.square().mean(-1)*mask).sum()/mask.sum()).sqrt().item()
    return first,sequence
def train_policy(horizon):
    x,y,m=x_all[train_rows],y_all[train_rows,:horizon],m_all[train_rows,:horizon]
    xv,yv,mv=x_all[val_rows].to(DEVICE),y_all[val_rows,:horizon].to(DEVICE),m_all[val_rows,:horizon].to(DEVICE)
    model=initialized_policy(horizon); before=copy.deepcopy(model.state_dict())
    optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE); generator=torch.Generator().manual_seed(SEED)
    history,best_val,best_weights=[],float("inf"),None; started=time.perf_counter()
    for update in range(1,UPDATES+1):
        batch=torch.randint(len(x),(BATCH_SIZE,),generator=generator)
        xb,yb,mb=x[batch].to(DEVICE),y[batch].to(DEVICE),m[batch].to(DEVICE)
        train_loss=learning_step(model,optimizer,xb,yb,mb)
        if update==1 or update%100==0 or update==UPDATES:
            model.eval()
            with torch.no_grad():
                tf,ts=native_errors(model(xb),yb,mb); vf,vs=native_errors(model(xv),yv,mv)
            history.append({"update":update,"masked_train_loss":train_loss,"train_first_error":tf,
                            "validation_first_error":vf,"train_sequence_RMSE":ts,"validation_sequence_RMSE":vs})
            if vf<best_val: best_val,best_weights=vf,copy.deepcopy(model.state_dict())
    model.load_state_dict(best_weights); model.eval()
    weight_change=sum((model.state_dict()[k]-before[k]).abs().sum().item() for k in before)
    assert weight_change>0 and np.isfinite(best_val)
    return {"model":model,"horizon":horizon,"stats":stats,"history":pd.DataFrame(history),
            "seconds":time.perf_counter()-started,"weight_change":weight_change,"best_val":best_val}
a,b=initialized_policy(1),initialized_policy(LONG_HORIZON)
assert all(torch.equal(a.trunk.state_dict()[k],b.trunk.state_dict()[k]) for k in a.trunk.state_dict())
assert torch.equal(a.head.weight,b.head.weight[:2]) and torch.equal(a.head.bias,b.head.bias[:2])
runs={h:train_policy(h) for h in HORIZONS}
display(pd.DataFrame([{"horizon":h,"validation_first_error":r["best_val"],"training_seconds":r["seconds"],
                       "weight_change_L1":r["weight_change"]} for h,r in runs.items()]).round(3))
fig,axes=plt.subplots(1,2,figsize=(10,3))
for h,run in runs.items():
    history=run["history"]
    axes[0].plot(history["update"],history.train_first_error,label=f"H={h}")
    axes[1].plot(history["update"],history.validation_first_error,label=f"H={h}")
for ax,title in zip(axes,["Training minibatch","Held-out validation"]):
    ax.set(title=title,xlabel="Optimizer updates",ylabel="First-action endpoint error [sim units]"); ax.legend(); ax.grid(alpha=.15)
plt.tight_layout(); plt.show()
print("Both checkpoints are selected by first-action validation error.")

### Task PAI3-T07 — Inspect Action Ambiguity · 4 min
**Do:** Run the real-data neighborhood diagnostic; compare observed targets with their mean.  
**Produce:** Action scatter and mean-to-label distance.  
**Check:** Nearby but different observations do not prove conditional multimodality.

In [ ]:
NEIGHBORS=32; reference_rows=np.flatnonzero(train_rows); query=np.flatnonzero(val_rows)[0]
distance=np.mean(((features[reference_rows]-features[query])/stats["x_std"])**2,axis=1)
neighbors=reference_rows[np.argsort(distance)[:NEIGHBORS]]; neighbor_actions=chunk_labels[neighbors,0]
local_mean=neighbor_actions.mean(0); closest_observed_action=float(np.linalg.norm(neighbor_actions-local_mean,axis=1).min())
fig,ax=plt.subplots(figsize=(5.3,3)); ax.scatter(neighbor_actions[:,0],neighbor_actions[:,1],c=CYAN,alpha=.6,label="Real neighboring labels")
ax.scatter(*local_mean,c=BLUE,marker="X",s=90,label="Local MSE mean")
ax.set(xlabel="Displacement x [sim units]",ylabel="Displacement y [sim units]"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show(); print(f"Mean to nearest observed action: {closest_observed_action:.3f} sim units")

### Task PAI3-T08 — Decode And Execute A Chunk · 6 min
**Do:** Complete TODO 4 and 5: restore absolute targets and select the correct indexed future action. Run both interface checks.  
**Produce:** A valid action sequence.  
**Check:** All future targets use the original chunk anchor; execution is not repeating one action.

In [ ]:
def decode_action_chunk(normalized_chunk,agent_anchor,stats):
    displacements=None  # TODO
    assert displacements is not None,"TODO 4: invert training action normalization."
    absolute_targets=None  # TODO
    assert absolute_targets is not None,"TODO 4: restore future absolute targets using the current anchor."
    return np.clip(absolute_targets,0,512).astype("float32")
@torch.no_grad()
def predict_chunk(run,previous_obs,current_obs):
    history=np.concatenate([previous_obs,current_obs]).astype("float32")
    normalized=(history-stats["x_mean"])/stats["x_std"]
    predicted=run["model"](torch.tensor(normalized[None],device=DEVICE)).cpu().numpy()[0]
    return decode_action_chunk(predicted,current_obs[:2],stats)
normalized_test=(chunk_labels[full_row]-stats["y_mean"])/stats["y_std"]
assert np.allclose(decode_action_chunk(normalized_test,agent_xy[full_row],stats),
                   np.clip(actions[full_row:full_row+LONG_HORIZON],0,512),atol=1e-4)
print("PASS: future targets reconstruct in the correct action interface.")

In [ ]:
def chunk_action_index(step_index,execute_prefix):
    chunk_index=None  # TODO
    assert chunk_index is not None,"TODO 5: index within the executed prefix."
    return int(chunk_index)
def choose_chunk_action(active_chunk,chunk_index):
    action=None  # TODO
    assert action is not None,"TODO 5: select the indexed future action, not always the first."
    return action
assert [chunk_action_index(t,4) for t in range(6)]==[0,1,2,3,0,1]
actual_chunk=np.clip(actions[full_row:full_row+LONG_HORIZON],0,512)
assert all(np.allclose(choose_chunk_action(actual_chunk,j),actual_chunk[j]) for j in range(LONG_HORIZON))
print("PASS: distinct future actions and receding-horizon execution.")

### Task PAI3-T09 — Evaluate Feedback And State Coverage · 8 min
**Do:** Run rollouts, metrics/GIFs, prefix comparison, and distribution-distance cells. Compare prefix 1 versus 4 using the SAME H=4 weights and reset seeds.  
**Produce:** Task progress, policy calls, and expert-versus-learner state-distance graphs.  
**Check:** Keep strict success separate from partial progress; nearest-data distance is not calibrated uncertainty.

In [ ]:
#@title Provided code — run this cell
def pack_observation(observation):
    packed=np.concatenate([observation["agent_pos"],observation["environment_state"]]).astype("float32")
    assert packed.shape==(18,) and np.isfinite(packed).all(); return packed
def make_env():
    return gym.make("gym_pusht/PushT-v0",obs_type="environment_state_agent_pos",render_mode="rgb_array",
                    max_episode_steps=200,visualization_width=256,visualization_height=256)
env=make_env(); observation,_=env.reset(seed=1001); current=pack_observation(observation)
logged,actual=object_xy[0].reshape(8,2),current[2:].reshape(8,2)
distances=lambda p:np.linalg.norm(p[:,None,:]-p[None,:,:],axis=-1)
assert np.allclose(distances(logged),distances(actual),atol=.003)
handedness=lambda p:np.linalg.det(np.stack([p[1]-p[0],p[2]-p[0]]))
assert handedness(logged)*handedness(actual)>0
checked=predict_chunk(runs[LONG_HORIZON],current,current)
assert checked.shape==(LONG_HORIZON,2) and env.action_space.contains(checked[0]); env.step(checked[0]); env.close()
offline=[]
for h,run in runs.items():
    with torch.no_grad():
        first,sequence=native_errors(run["model"](x_all[test_rows].to(DEVICE)),y_all[test_rows,:h].to(DEVICE),m_all[test_rows,:h].to(DEVICE))
    offline.append({"prediction_horizon":h,"test_first_action_error":first,"test_sequence_RMSE":sequence})
offline_df=pd.DataFrame(offline); display(offline_df.round(3))
print("First-action error is common; sequence RMSEs refer to different horizon-specific prediction tasks.")
EVAL_EPISODES=int(os.environ.get("AG_LAB3_EVAL_EPISODES","8")); MAX_STEPS=int(os.environ.get("AG_LAB3_MAX_STEPS","200"))
assert EVAL_EPISODES>=1 and MAX_STEPS>=1
evaluation_seeds=list(range(1001,1001+EVAL_EPISODES)); conditions=[(1,1),(LONG_HORIZON,1),(LONG_HORIZON,LONG_HORIZON)]
def rollout(run,seed,execute_prefix,record=False):
    assert 1<=execute_prefix<=run["horizon"]
    env=make_env(); observation,_=env.reset(seed=seed); current=pack_observation(observation); previous=current.copy()
    initial_coverage=float(env.unwrapped._get_coverage()); final_coverage=initial_coverage; peak_coverage=initial_coverage; success=False
    histories,frames,commands=[],[],[]; active_chunk=None; start=time.perf_counter()
    for t in range(MAX_STEPS):
        chunk_index=chunk_action_index(t,execute_prefix)
        if chunk_index==0:
            histories.append(np.concatenate([previous,current]).copy()); active_chunk=predict_chunk(run,previous,current)
        action=choose_chunk_action(active_chunk,chunk_index); commands.append(action.copy())
        if record and t%2==0: frames.append(PILImage.fromarray(env.render()))
        observation,_,terminated,truncated,info=env.step(action)
        final_coverage=float(info["coverage"]); peak_coverage=max(peak_coverage,final_coverage); success=success or bool(info["is_success"])
        previous,current=current.copy(),pack_observation(observation)
        if terminated or truncated: break
    if record: frames.append(PILImage.fromarray(env.render()))
    env.close()
    return {"prediction_horizon":run["horizon"],"execute_prefix":execute_prefix,"seed":seed,"success":int(success),
            "initial_coverage":initial_coverage,"final_coverage":final_coverage,"peak_coverage":peak_coverage,
            "coverage_gain":final_coverage-initial_coverage,"steps":t+1,"policy_calls":len(histories),"seconds":time.perf_counter()-start,
            "_histories":np.asarray(histories,dtype="float32"),"_commands":np.asarray(commands)},frames
records,visited_histories,saved_frames=[],{},{}
for h,prefix in conditions:
    visited_histories[(h,prefix)]=[]
    for seed in evaluation_seeds:
        row,frames=rollout(runs[h],seed,prefix,record=(seed==evaluation_seeds[0]))
        visited_histories[(h,prefix)].append(row.pop("_histories")); commands=row.pop("_commands")
        if seed==evaluation_seeds[0]:
            saved_frames[(h,prefix)]=frames; np.save(OUT/f"actual_commands_H{h}_prefix{prefix}.npy",commands)
        records.append(row)
results=pd.DataFrame(records); print(f"Executed {len(results)} real rollouts: two models, three execution conditions.")

In [ ]:
def wilson_interval(successes,n,z=1.96):
    p=successes/n; denominator=1+z*z/n; center=(p+z*z/(2*n))/denominator
    radius=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/denominator; return center-radius,center+radius
rows=[]
for (h,prefix),group in results.groupby(["prediction_horizon","execute_prefix"]):
    low,high=wilson_interval(int(group.success.sum()),len(group))
    rows.append({"prediction_horizon":h,"execute_prefix":prefix,"rollouts":len(group),"success_rate":group.success.mean(),
                 "CI95_low":low,"CI95_high":high,"mean_final_coverage":group.final_coverage.mean(),"mean_peak_coverage":group.peak_coverage.mean(),
                 "mean_net_coverage_gain":group.coverage_gain.mean(),"mean_policy_calls":group.policy_calls.mean()})
summary=pd.DataFrame(rows).merge(offline_df,on="prediction_horizon"); display(summary.round(3))
labels=[f"H={int(r.prediction_horizon)}, prefix={int(r.execute_prefix)}" for _,r in summary.iterrows()]
fig,axes=plt.subplots(1,3,figsize=(12,3.3))
axes[0].bar(offline_df.prediction_horizon.astype(str),offline_df.test_first_action_error,color=BLUE)
axes[0].set(xlabel="Prediction horizon",ylabel="First-action endpoint error [sim units]",title="Common offline task")
axes[1].bar(labels,summary.success_rate,color=CYAN,yerr=np.vstack([summary.success_rate-summary.CI95_low,summary.CI95_high-summary.success_rate]),capsize=3)
axes[1].set(ylim=(0,1.05),title="Task success + 95% Wilson interval"); axes[1].tick_params(axis="x",labelrotation=20)
axes[2].bar(labels,summary.mean_net_coverage_gain,color=[BLUE,CYAN,NAVY]); axes[2].axhline(0,color=SILVER,lw=1)
axes[2].set(ylabel="Final − initial goal coverage",title="Net task progress"); axes[2].tick_params(axis="x",labelrotation=20)
plt.tight_layout(); plt.show()
for (h,prefix),frames in saved_frames.items():
    gif=OUT/f"actual_rollout_H{h}_prefix{prefix}.gif"
    frames[0].save(gif,save_all=True,append_images=frames[1:],duration=200,loop=0)
    print(f"Actual rollout H={h}, prefix={prefix}"); display(Image(filename=str(gif)))
print("Eight resets per condition are a teaching sample, not a deployment estimate.")

In [ ]:
paired=results[results.prediction_horizon==LONG_HORIZON].pivot(index="seed",columns="execute_prefix",values=["final_coverage","policy_calls"])
fig,axes=plt.subplots(1,2,figsize=(10,3))
paired["final_coverage"].plot.bar(ax=axes[0],color=[BLUE,CYAN]); paired["policy_calls"].plot.bar(ax=axes[1],color=[BLUE,CYAN])
axes[0].set(ylabel="Final goal coverage",ylim=(0,1.05),title="Same initial conditions")
axes[1].set(ylabel="Policy calls",title="Feedback / computation trade-off")
for ax in axes: ax.legend(title="Executed prefix",fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
training_reference=x_all[train_rows]
def nearest_training_distance(raw_queries):
    queries=torch.tensor((raw_queries-stats["x_mean"])/stats["x_std"],dtype=torch.float32)
    return (torch.cdist(queries,training_reference).min(1).values/math.sqrt(36)).numpy()
expert_rows=features[val_rows]; expert_sample=expert_rows[np.linspace(0,len(expert_rows)-1,64,dtype=int)]
expert_distance=nearest_training_distance(expert_sample); shift_records=[]
fig,axes=plt.subplots(1,len(conditions),figsize=(12,3),squeeze=False)
for ax,(h,prefix) in zip(axes[0],conditions):
    learner_rows=np.concatenate(visited_histories[(h,prefix)])
    learner_distance=nearest_training_distance(learner_rows[np.linspace(0,len(learner_rows)-1,64,dtype=int)])
    ax.hist(expert_distance,bins=12,alpha=.6,color=CYAN,label="Held-out expert"); ax.hist(learner_distance,bins=12,alpha=.6,color=BLUE,label="Learner visited")
    ax.set(title=f"H={h}, prefix={prefix}",xlabel="Nearest training history distance",ylabel="Count"); ax.legend(fontsize=7)
    for source,values in [("held_out_expert",expert_distance),("learner_visited",learner_distance)]:
        shift_records.extend({"prediction_horizon":h,"execute_prefix":prefix,"source":source,"distance":float(v)} for v in values)
shift_df=pd.DataFrame(shift_records); plt.tight_layout(); plt.show()
display(shift_df.groupby(["prediction_horizon","execute_prefix","source"]).distance.median().round(3))

### Task PAI3-T10 — Save And Explain · 4 min
**Do:** Complete the five short answer fields using your figures, including which expert labels DAgger would require. Run export.  
**Produce:** Checkpoints, CSV metrics, protocol, and skills evidence.  
**Check:** Reloaded predictions match. This is chunked BC, not DAgger or VLA training.

In [ ]:
answers={"offline_vs_closed_loop":"Use common first-action error and task metrics.",
           "action_ambiguity":"Use the mean-to-observed-label distance and explain its limitation.",
           "execution_horizon":"Compare progress and policy calls for the SAME H=4 weights.",
           "distribution_shift":"Compare expert and learner distances; do not call distance a safety guarantee.",
           "dagger_boundary":"Which expert labels would DAgger require?"}
for h,run in runs.items():
    checkpoint={"state_dict":run["model"].state_dict(),"prediction_horizon":h,"architecture":[36,128,128,2*h],
                "normalization":{k:v.tolist() for k,v in stats.items()},"history_frames":2,
                "action_semantics":"future absolute targets relative to CURRENT agent anchor",
                "padding":"masked after episode end; mask-weighted sequence MSE","train_episode_ids":train_ids.tolist(),
                "dataset_revision":REVISION,"learning_rate":LEARNING_RATE,"optimizer_updates":UPDATES,"seed":SEED}
    file=OUT/f"chunk_policy_H{h}.pt"; torch.save(checkpoint,file)
    saved=torch.load(file,map_location="cpu",weights_only=True); reloaded=Policy(saved["prediction_horizon"]).eval()
    reloaded.load_state_dict(saved["state_dict"]); sample=x_all[test_rows][:4]
    with torch.no_grad(): assert torch.allclose(reloaded(sample),run["model"](sample.to(DEVICE)).cpu(),atol=1e-6)
    run["history"].to_csv(OUT/f"training_H{h}.csv",index=False)
results.to_csv(OUT/"closed_loop_rollouts.csv",index=False); summary.to_csv(OUT/"comparison.csv",index=False)
offline_df.to_csv(OUT/"offline_first_and_sequence_errors.csv",index=False); shift_df.to_csv(OUT/"distribution_shift.csv",index=False)
protocol={"training_episodes":TRAIN_EPISODES,"prediction_horizons":HORIZONS,"optimizer_updates":UPDATES,"learning_rate":LEARNING_RATE,
          "matched_initial_trunk_and_first_action_head":True,"evaluation_seeds":evaluation_seeds,"execution_conditions":conditions,
          "max_steps":MAX_STEPS,"success":"native Gym PushT coverage > 0.95","checkpoint_selection":"first-action validation error",
          "common_offline_metric":"first-action endpoint error in simulation units","sequence_metric":"horizon-specific masked per-component RMSE"}
(OUT/"experiment_protocol.json").write_text(json.dumps(protocol,indent=2))
skills=[{"skill":"Episode-safe future labels","evidence":"data_manifest.json","check":"current anchor, same episode, padding mask"},
        {"skill":"Train-only normalization","evidence":"chunk_policy_H*.pt","check":"same 100 training episodes, common statistics"},
        {"skill":"Masked sequence learning","evidence":"training_H*.csv","check":"padding-invariance test and real weight updates"},
        {"skill":"Execute predicted action chunks","evidence":"actual_commands_H*_prefix*.npy","check":"indexed future targets, absolute command interface"},
        {"skill":"Receding-horizon execution","evidence":"closed_loop_rollouts.csv","check":"same model and seeds; prefix1 versus prefixH"},
        {"skill":"Learner-induced shift","evidence":"distribution_shift.csv","check":"actual learner versus expert histories"},
        {"skill":"Interpret ambiguity","evidence":"answers.json","check":"nearby states alone do not prove conditional multimodality"},
        {"skill":"Executable checkpoints","evidence":"chunk_policy_H*.pt","check":"reload predictions match"}]
pd.DataFrame(skills).to_csv(OUT/"skills_evidence.csv",index=False)
boundaries={"implemented":"deterministic H-step behavioral cloning, real gradient updates, masked future labels, receding-horizon execution",
            "not_implemented":["DAgger expert relabeling","mixture/diffusion policy training","foundation VLA fine-tuning","real-robot safety validation"]}
(OUT/"capability_boundaries.json").write_text(json.dumps(boundaries,indent=2)); (OUT/"answers.json").write_text(json.dumps(answers,indent=2))
print("PASS: trained checkpoints reload with matching predictions."); print("Saved:",", ".join(p.name for p in sorted(OUT.iterdir())))

<details><summary>Sources and experiment contract</summary>

ArtificialGate, *Physical AI Essentials*, Chapter3: Behavioral Cloning; Action Chunks; Normalization; Splits; Evaluation.
[LeRobot PushT keypoints](https://huggingface.co/datasets/lerobot/pusht_keypoints) and [Gym PushT](https://github.com/huggingface/gym-pusht), Apache-2.0: simulation demonstrations, Pymunk evaluation, not real-robot measurements or the website's PhysX simulator.
[Ross, Gordon & Bagnell2011](https://proceedings.mlr.press/v15/ross11a.html): learner-induced shift and DAgger. This exercise does not query expert labels on learner-visited states, so it is not DAgger.

Future observations, rewards and success flags never enter the policy. Both models use the same episodes, normalization, common initial layers and update count. H=4 has a slightly larger output head; equal updates are not identical FLOPs. No improvement is guaranteed. Deterministic MSE policies can fail despite low offline error.
</details>